# Agglomerative Cluster Analysis — EFIplus_medit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/FernandaChacara/AVCAD/blob/main/Exercise7/EFIplus_cluster_analysis_exercise.ipynb)

This notebook solves the exercise using the `EFIplus_medit.zip` dataset:

1. Agglomerative cluster analysis using different linkage methods, based on quantitative environmental variables, using only sites from the **Douro** and **Tejo** basins.  
2. Heatmap and dendrogram by clustering the rows/sites using **average linkage**.  
3. Dendrogram clustering the environmental variables/columns using **average linkage**, followed by a short discussion on variable selection for regression-based analysis.

The quantitative environmental variables used are the same as in the previous exercise:

`Altitude`, `Actual_river_slope`, `Elevation_mean_catch`, `prec_ann_catch`, `temp_ann`, `temp_jan`, `temp_jul`.


## 1. Install and import libraries

The main libraries used are:

- `pandas` and `numpy` for data handling;
- `scikit-learn` for standardisation;
- `scipy` for hierarchical clustering and dendrograms;
- `matplotlib` and `seaborn` for visualisation.


In [ ]:
# Import libraries
import os
import zipfile
import glob
import warnings

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import pdist

warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (10, 6)
sns.set_context('notebook')


## 2. Load the dataset

This cell tries to find the dataset automatically. It works in two ways:

1. If the ZIP file is already in the Colab/session folder, it will be used directly.
2. If it is not available, the notebook downloads it from the GitHub repository.

If the automatic download fails, upload the file manually using the Colab file panel.


In [ ]:
# Dataset file names accepted by this notebook
possible_zip_names = [
    'EFIplus_medit.zip',
    'EFIplus_medit (2).zip'
]

zip_path = None

# Check whether the file already exists locally
for name in possible_zip_names:
    if os.path.exists(name):
        zip_path = name
        break

# If not found, try to download from GitHub
if zip_path is None:
    print('ZIP file not found locally. Trying to download it from GitHub...')
    !wget -O "EFIplus_medit.zip" "https://github.com/FernandaChacara/AVCAD/raw/main/Exercise7/EFIplus_medit%20(2).zip"
    if os.path.exists('EFIplus_medit.zip'):
        zip_path = 'EFIplus_medit.zip'

print('Using ZIP file:', zip_path)

# Extract ZIP file
extract_dir = 'EFIplus_medit_extracted'
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_dir)

# Show extracted files
extracted_files = glob.glob(extract_dir + '/**/*', recursive=True)
print('Extracted files:')
for file in extracted_files:
    if os.path.isfile(file):
        print(file)


## 3. Read the data file

The ZIP may contain a CSV, TXT, DAT or Excel file. The code below searches for a readable table and loads it into a dataframe.


In [ ]:
# Find candidate data files
candidate_files = []
for ext in ['*.csv', '*.txt', '*.dat', '*.xlsx', '*.xls']:
    candidate_files.extend(glob.glob(extract_dir + '/**/' + ext, recursive=True))

print('Candidate files:', candidate_files)

# Function to read different table formats
def read_table_auto(path):
    if path.endswith(('.xlsx', '.xls')):
        return pd.read_excel(path)
    
    # Try common separators
    for sep in [';', ',', '\t']:
        try:
            df_try = pd.read_csv(path, sep=sep)
            if df_try.shape[1] > 1:
                return df_try
        except Exception:
            pass
    
    # Last attempt: automatic separator inference
    return pd.read_csv(path, sep=None, engine='python')

# Load the first valid table found
df = None
for file in candidate_files:
    try:
        temp_df = read_table_auto(file)
        if temp_df.shape[0] > 0 and temp_df.shape[1] > 1:
            df = temp_df
            print('Loaded file:', file)
            break
    except Exception as e:
        print('Could not read:', file, e)

if df is None:
    raise ValueError('No valid data table was found inside the ZIP file.')

print('Dataset shape:', df.shape)
df.head()


## 4. Select Douro and Tejo sites and quantitative environmental variables

Only sites from the **Douro** and **Tejo** basins are used in this exercise.

Before clustering, the environmental variables are standardised. This is necessary because the variables are measured in different units, for example altitude, slope, precipitation and temperature. Without standardisation, variables with larger numerical scales could dominate the clustering results.


In [ ]:
# Environmental variables used in the previous exercise
environmental_vars = [
    'Altitude',
    'Actual_river_slope',
    'Elevation_mean_catch',
    'prec_ann_catch',
    'temp_ann',
    'temp_jan',
    'temp_jul'
]

# Check required columns
required_cols = ['Catchment_name'] + environmental_vars
missing_cols = [col for col in required_cols if col not in df.columns]

if missing_cols:
    print('Available columns:')
    print(df.columns.tolist())
    raise ValueError(f'Missing columns in dataset: {missing_cols}')

# Filter only Douro and Tejo basins
data = df[df['Catchment_name'].isin(['Douro', 'Tejo'])].copy()

# Keep only complete cases for the selected variables
data = data.dropna(subset=environmental_vars + ['Catchment_name'])

print('Filtered dataset shape:', data.shape)
print(data['Catchment_name'].value_counts())

# Create a site label for plotting
if 'Site_code' in data.columns:
    data['Site_label'] = data['Site_code'].astype(str)
elif 'Site' in data.columns:
    data['Site_label'] = data['Site'].astype(str)
else:
    data['Site_label'] = data.index.astype(str)

# Matrix of environmental variables
X = data[environmental_vars].copy()

# Standardise variables
scaler = StandardScaler()
X_scaled = pd.DataFrame(
    scaler.fit_transform(X),
    columns=environmental_vars,
    index=data['Site_label']
)

X_scaled.head()


## 5. Agglomerative cluster analysis using different linkage methods

Agglomerative clustering starts with each site as its own cluster and progressively joins the most similar clusters.

Here, different linkage methods are compared:

- **single linkage**: joins clusters based on the closest pair of observations;
- **complete linkage**: joins clusters based on the farthest pair of observations;
- **average linkage**: joins clusters based on the average distance between observations;
- **ward linkage**: minimises within-cluster variance.

The Euclidean distance is used because the variables have been standardised.


In [ ]:
# Compute Euclidean distances between sites
site_distances = pdist(X_scaled, metric='euclidean')

# Linkage methods to compare
linkage_methods = ['single', 'complete', 'average', 'ward']

# Plot dendrograms for each linkage method
for method in linkage_methods:
    Z = linkage(site_distances, method=method)
    
    plt.figure(figsize=(14, 6))
    dendrogram(
        Z,
        labels=X_scaled.index.tolist(),
        leaf_rotation=90,
        leaf_font_size=7,
        color_threshold=None
    )
    plt.title(f'Agglomerative clustering of Douro and Tejo sites — {method} linkage')
    plt.xlabel('Sites')
    plt.ylabel('Euclidean distance')
    plt.tight_layout()
    plt.show()


### Brief interpretation of the linkage methods

The different linkage methods can produce different dendrogram structures because they define the distance between clusters differently. Single linkage is more sensitive to chaining effects, while complete linkage tends to form more compact clusters. Average linkage is often a balanced option because it considers the average distance between all observations in two clusters. Ward linkage usually produces compact groups by minimising within-cluster variance.


## 6. Heatmap and dendrogram clustering rows/sites using average linkage

The heatmap below shows the standardised environmental values for each site. Rows are clustered using **average linkage**, as requested in the exercise.

The colour pattern helps identify groups of sites with similar environmental profiles.


In [ ]:
# Row colours by catchment, to help visual interpretation
catchment_palette = dict(zip(
    data['Catchment_name'].unique(),
    sns.color_palette('Set2', n_colors=data['Catchment_name'].nunique())
))

row_colors = data.set_index('Site_label')['Catchment_name'].map(catchment_palette)

# Heatmap with row clustering using average linkage
sns.clustermap(
    X_scaled,
    method='average',
    metric='euclidean',
    row_cluster=True,
    col_cluster=False,
    row_colors=row_colors,
    cmap='vlag',
    figsize=(12, 14),
    yticklabels=True,
    xticklabels=True
)

plt.suptitle('Heatmap of environmental variables — rows clustered with average linkage', y=1.02)
plt.show()


### Separate dendrogram of sites using average linkage

The following dendrogram shows only the site clustering using average linkage.


In [ ]:
# Dendrogram using average linkage for sites
Z_average_sites = linkage(site_distances, method='average')

plt.figure(figsize=(14, 7))
dendrogram(
    Z_average_sites,
    labels=X_scaled.index.tolist(),
    leaf_rotation=90,
    leaf_font_size=7
)
plt.title('Dendrogram of Douro and Tejo sites — average linkage')
plt.xlabel('Sites')
plt.ylabel('Euclidean distance')
plt.tight_layout()
plt.show()


## 7. Dendrogram clustering the environmental variables instead of the sites

To cluster the environmental variables, the dataframe is transposed using `.T`. This makes the variables become the rows of the matrix.

This is useful because it shows which variables have similar patterns across sites. Variables that cluster very closely may carry similar information and may be correlated.


In [ ]:
# Transpose dataframe: variables become rows and sites become columns
X_variables = X_scaled.T

# Compute distances between environmental variables
variable_distances = pdist(X_variables, metric='euclidean')

# Average linkage clustering of variables
Z_average_variables = linkage(variable_distances, method='average')

plt.figure(figsize=(10, 6))
dendrogram(
    Z_average_variables,
    labels=X_variables.index.tolist(),
    leaf_rotation=45,
    leaf_font_size=10
)
plt.title('Dendrogram of environmental variables — average linkage')
plt.xlabel('Environmental variables')
plt.ylabel('Euclidean distance')
plt.tight_layout()
plt.show()


## 8. Discussion: how variable clustering helps regression-based analysis

The dendrogram that clusters the environmental variables helps identify variables with similar behaviour across the Douro and Tejo sites. When two or more variables cluster very closely, this suggests that they may contain overlapping information. In a regression-based analysis, this is important because highly similar or strongly correlated predictors can create multicollinearity.

For example, if `temp_ann`, `temp_jan` and `temp_jul` appear close together in the dendrogram, they may represent related aspects of the same temperature gradient. In that case, it may not be necessary to include all of them in the same regression model. The researcher could select one representative temperature variable, or compare alternative models using different temperature predictors.

The same reasoning applies to altitude-related variables, such as `Altitude` and `Elevation_mean_catch`. If they cluster closely, they may express a similar environmental gradient. Keeping both variables in a regression model could make interpretation more difficult because their effects may not be independent.

Therefore, clustering variables is a useful exploratory step before regression. It helps to detect redundancy among predictors, guide variable selection, reduce model complexity and improve the interpretability of ecological models.


## 9. Final notes

This notebook completes the three requested tasks:

1. Agglomerative cluster analysis with different linkage methods for Douro and Tejo sites.
2. Heatmap and dendrogram clustering sites using average linkage.
3. Dendrogram clustering environmental variables using average linkage, with discussion on its usefulness for regression-based analysis.
